# 13주차. 통합 정책 시뮬레이션 워크숍

## 오늘의 목표

이번 시간에는 새 이론을 배우지 않는다. 한 학기 동안 배운 5개 도구(인과 분석·시스템 사고·베이지안·시나리오·몬테카를로)를 **하나의 정책 사례**에 차례대로 적용한다.

같은 문제를 다섯 가지 렌즈로 본다는 뜻이다. 어떤 렌즈가 어떤 답을 주는지, 그리고 답이 충돌할 때 기획자는 무엇을 보아야 하는지 익힌다.

수업이 끝나면 다음을 할 수 있어야 한다.

1. 한 정책 사례에 5개 도구를 적용해 5가지 인사이트를 얻는다.
2. 도구별 결과가 서로 어떻게 보완·충돌하는지 설명한다.
3. 다섯 결과를 하나의 권고안으로 종합한다.
4. 통합 기획서의 골격을 워크숍 결과로 채운다.

> **한 줄 요약**: 같은 문제를 다섯 렌즈로 보고, 다섯 답을 하나의 권고로 묶는 연습이다.


---

## 1. 오늘의 사례: 청년 주거 지원금 정책

지방 정부가 **연 1,000억 원**을 청년 주거 지원에 배정하기로 했다. 기획자에게 다음 질문이 떨어졌다.

> "지원금을 **현금 직접 지원**으로 줄까, **임대료 보조 (월세 지원)**로 줄까, **주거 인프라 (공공임대 신축)**에 투자할까? 효과·리스크·재정 부담은 어떻게 다른가?"

이 질문에는 정답이 하나가 아니다. 도구마다 답이 다르다. 오늘은 **5개 도구로 각각 답을 얻고, 마지막에 하나의 권고로 묶는다**.

| 단계 | 도구 | 묻는 질문 | 활용 주차 |
|------|------|---------|---------|
| 1 | 인과 분석 (DAG) | 무엇이 무엇을 일으키는가? 숨은 원인은? | 4주차 |
| 2 | 시스템 다이내믹스 (CLD) | 피드백 루프와 시간 지연은? | 5주차 |
| 3 | 베이지안 의사결정 | 새 정보가 들어오면 믿음을 어떻게 갱신하나? | 6주차 |
| 4 | 시나리오 플래닝 | 어떤 미래에서도 살아남는 전략은? | 7주차 |
| 5 | 몬테카를로 시뮬레이션 | 결과 분포는? 최악·최선은? | 9주차 |


---

## 2. 공통 라이브러리

다섯 단계 모두에서 쓰는 기본 라이브러리를 한 번에 불러온다.


In [ ]:
# 기본 라이브러리
import matplotlib.font_manager as fm
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# 한글 폰트 설정
korean_fonts = ["AppleGothic", "Malgun Gothic", "맑은 고딕", "NanumGothic"]
available = {f.name for f in fm.fontManager.ttflist}
for font in korean_fonts:
    if font in available:
        plt.rcParams["font.family"] = font
        break
plt.rcParams["axes.unicode_minus"] = False

# 흑백 기조
plt.rcParams["axes.facecolor"] = "white"
plt.rcParams["axes.edgecolor"] = "black"

np.random.seed(13)  # 재현성


---

## 3. 1단계: 인과 분석 (DAG) — 무엇이 무엇을 일으키나

먼저 정책 효과의 인과 구조를 그림으로 정리한다. 4주차에서 배운 DAG로 **숨은 원인 (교란변수)** 과 **합류점 (Collider)** 을 점검한다.

청년 주거 정책의 원인-결과 지도(예시):

```
   가구 소득 ──┐
              ↓
지원금 정책 → 주거 비용 ──→ 청년 주거 안정 ←── 주거 인프라
              ↑                  ↑
              ├── 임대 시장 가격 ──┘
              ↑
        지역 경기 (숨은 원인)
```

**기획자의 질문**

> 지원금을 늘리면 주거 안정이 올라간다는 단순 인과를 가정해도 될까?
> 임대 시장이 동시에 가격을 올리면 효과가 상쇄될 수 있지 않나?

**AI에게 보낼 요청**

```text
청년 주거 지원금 정책의 DAG를 그려줘.
변수는 다음 7개를 포함해야 해.
- 지원금 정책 (개입 변수)
- 주거 비용
- 청년 주거 안정 (결과)
- 가구 소득 (교란변수 후보)
- 임대 시장 가격 (매개·교란 후보)
- 주거 인프라 (대안 정책)
- 지역 경기 (숨은 원인)

각 화살표 옆에 "양(+)" 또는 "음(-)" 영향을 적어주고,
교란변수와 매개변수가 어디인지 표시해줘.
NetworkX와 matplotlib으로 그리는 코드를 만들어줘.
```

AI가 준 코드를 아래 빈 셀에 붙여 넣고 실행한다. 실행 후 **숨은 원인이 결과 해석을 어떻게 흔들 수 있는지** 한 줄로 메모한다.


---

## 4. 2단계: 시스템 다이내믹스 (CLD) — 피드백 루프와 시간 지연

DAG는 정태적이다. 시스템 사고는 **시간이 흐르면 어떻게 변하는가**를 본다. 5주차 CLD로 **강화 루프 (R)** 와 **균형 루프 (B)** 를 그린다.

청년 주거 정책의 피드백 구조(예시):

```
지원금 ↑ → 청년 유입 ↑ → 임대 수요 ↑ → 임대료 ↑ → 지원금 효과 ↓   (R: 자기상쇄 강화)
                                              ↓
                       공공임대 공급 ↑ ← 정부 대응 ←┘                (B: 균형)
```

**핵심 직관**

- **현금 지원**은 빠르지만 **R 루프** 때문에 임대료가 올라 효과가 1~2년 안에 잠식된다.
- **공공임대**는 느리지만 **B 루프**로 시장 가격을 누른다 (지연 3~5년).
- 어느 쪽이 옳다는 답은 없다. **시점에 따라 다르다**.

**AI에게 보낼 요청**

```text
청년 주거 지원금 정책의 인과 루프 다이어그램(CLD)을 그려줘.
- 강화 루프 R1: 지원금 → 청년 유입 → 임대 수요 → 임대료 → 지원금 효과 잠식
- 균형 루프 B1: 임대료 상승 → 공공임대 공급 확대 → 임대료 안정
- 시간 지연 표시: 정부 대응 (1년), 공공임대 건설 (3~5년)

또한 5년 동안의 단순 시뮬레이션을 만들어줘.
- 변수: 지원금 정책 강도 (현금 vs 인프라 비율)
- 출력: 5년간 임대료, 청년 주거 안정 지수 변화
- matplotlib 흑백 라인 차트로 표시.
```

AI 코드 실행 후, **현금 vs 인프라 비율을 바꿨을 때 5년 후 결과가 어떻게 갈리는지** 한 줄 메모.


---

## 5. 3단계: 베이지안 의사결정 — 새 정보로 믿음 갱신

지원금 정책이 효과적일 확률을 처음에는 잘 모른다. **사전확률 (Prior)** 로 시작해서, 1년 시범사업 결과 (가능도, Likelihood)가 들어오면 **사후확률 (Posterior)** 로 갱신한다.

**시나리오**

- 사전 믿음: "현금 지원이 효과적일 확률 50% (반반이다)"
- 1년 후 시범사업 결과: 청년 주거 안정 지수 **+8% 상승** (기준 +5%)
- 가능도: 효과적일 때 +8% 이상 나올 확률 70%, 비효과적일 때는 30%

**계산**

```
Posterior = Prior × Likelihood / Evidence
         = 0.50 × 0.70 / (0.50 × 0.70 + 0.50 × 0.30)
         = 0.35 / 0.50
         = 0.70   →  믿음이 50% → 70%로 올라감
```

**기획자의 결정**

- 70% 확신이면 본사업으로 확대할 만하다.
- 50%였으면 시범 1년 더 연장이 합리적이다.
- 베이지안의 핵심: **결정을 한 번에 내리지 않는다. 새 데이터로 갱신한다**.

**AI에게 보낼 요청**

```text
3개 정책 옵션(현금·임대보조·인프라) 각각에 대해 베이지안 업데이트를 적용해줘.
- 사전확률: 모두 동일하게 1/3
- 1년 시범 결과 (가상):
  - 현금: 안정 지수 +8% (가능도 0.6)
  - 임대보조: +6% (가능도 0.5)
  - 인프라: +3% (단, 5년 후 +12% 예상, 가능도 0.4)
- 사후확률을 표로 출력하고, 의사결정 권고를 한 줄로 적어줘.
- 단, 인프라는 시간 지연이 있으니 베이지안 결과만으로 판단하지 말라는 경고도 포함해.
```


---

## 6. 4단계: 시나리오 플래닝 — 어떤 미래에서도 살아남나

베이지안은 "현재 시점 결정"을 돕는다. 시나리오는 "**여러 미래**에 대비한다". 7주차에서 배운 2축 매트릭스로 4개 시나리오를 만든다.

**핵심 불확실성 2개**

- 가로축: **금리 환경** (저금리 ↔ 고금리)
- 세로축: **청년 인구 유입** (증가 ↔ 감소)

```
청년 유입 ↑
        |
   ★ 호황   |   ★ 인구만 늘고 돈줄 막힘
   (현금 효과 ↑)|  (현금 부담 ↑, 인프라 우선)
        |
--------+--------→ 금리
        |
   ★ 인구 유출  |   ★ 빙하기
   (인프라 사업)|  (모든 정책 효과 ↓)
        |
청년 유입 ↓
```

**4개 시나리오별 권고**

| 시나리오 | 추천 정책 비중 |
|---------|--------------|
| 호황 (저금리·유입↑) | 현금 50% + 임대보조 30% + 인프라 20% |
| 인구 유입·고금리 | 임대보조 50% + 인프라 50% |
| 인구 유출·저금리 | 인프라 80% (장기 자산 확보) |
| 빙하기 (고금리·유출) | 인프라 30% + 예비비 70% (보수적) |

**강건한 (Robust) 전략**

> 4개 시나리오 모두에서 작동하는 비중은? — 인프라 30% 이상 + 현금 비중 가변. **인프라가 모든 미래의 공통 분모.**

**AI에게 보낼 요청**

```text
위 4시나리오 매트릭스를 시각화하는 Python 코드를 만들어줘.
- 흰 바탕, 검은 점, 회색 점선
- 각 분면에 시나리오 이름과 추천 정책 비중을 한글로 표시
- 마지막에 "모든 시나리오에서 작동하는 강건한 전략은 무엇인가?"라는 질문을 출력.
```


---

## 7. 5단계: 몬테카를로 시뮬레이션 — 결과 분포와 최악·최선

지금까지는 점 추정과 시나리오였다. 마지막은 **확률 분포**다. 9주차에서 배운 몬테카를로로 1만 번 굴려본다.

**모델**

- 입력 1: 청년 유입 증가율 ~ 정규(평균 5%, 표준편차 3%)
- 입력 2: 임대료 상승률 ~ 삼각(최저 1%, 최빈 4%, 최고 10%)
- 입력 3: 정책 효과 계수 ~ 정규(평균 0.6, 표준편차 0.2)
- 출력: 5년 후 청년 주거 안정 지수 (목표 +20%)

**1만 번 시뮬레이션 결과(예시)**

- 평균: +18%
- 5분위: +8% (최악)
- 95분위: +28% (최선)
- 목표(+20%) 달성 확률: **42%**

**기획자의 해석**

- 평균이 +18%라도 **목표 달성 확률은 42%** — 절반 이하.
- 5분위 +8%가 정치적으로 받아들여질지 미리 점검해야 한다.
- 정책을 바꾸지 않으면 목표 미달 확률이 58%다.

**AI에게 보낼 요청**

```text
청년 주거 지원금 정책의 5년 후 안정 지수를 몬테카를로 시뮬레이션해줘.
- 시행 횟수: 10000
- 입력 분포: 위에 적은 3개 변수
- 출력:
  1) 결과 분포 히스토그램 (흑백)
  2) 평균, 중앙값, 5분위, 95분위, 목표 +20% 달성 확률
  3) 각 입력 변수의 민감도 (토네이도 차트)
- 결과를 한 줄로 해석해줘.
```


---

## 8. 통합 인사이트: 다섯 답을 하나의 권고로

5개 도구는 서로 다른 답을 줬다. 기획자의 일은 **통합**이다.

| 도구 | 답 |
|------|---|
| DAG | 단순 인과 가정은 위험. 임대 시장·소득이 결과를 흔든다 |
| CLD | 현금은 1~2년 안에 자기상쇄. 인프라는 3~5년 지연되지만 지속 |
| 베이지안 | 시범사업 결과로 본사업 결정 (현재 시점 권고) |
| 시나리오 | 인프라 30% 이상은 모든 미래의 공통 분모 |
| 몬테카를로 | 현행 안 그대로면 목표 달성 확률 42% — 부족 |

**통합 권고안 (예시)**

> **"3년 단계적 비중 조정 전략"**
> - 1년차: 현금 60% + 임대보조 30% + 인프라 10% (즉시 효과)
> - 2년차: 현금 40% + 임대보조 30% + 인프라 30% (시장 반응 관찰)
> - 3년차: 현금 20% + 임대보조 30% + 인프라 50% (장기 자산화)
> - 매년 KPI(11주차)와 베이지안 업데이트로 비중 재조정 (12주차 적응적 기획)

**핵심 통찰**

> 한 도구만으로 답을 내면 위험하다. **다섯 도구의 답이 충돌하는 지점이 진짜 의사결정 지점**이다.


---

## 9. 학생 실습: 자기 주제에 5개 도구 적용

기말 프로젝트 주제를 골라 위 5단계를 적용한다.

**예시 주제**

- 전기차 충전 인프라 확충
- 지역 관광 활성화
- 학교 급식 친환경 전환
- 고령자 디지털 교육 확대

**진행 순서**

1. 주제 1개 선택 → 정책 옵션 3개 정의 (예: A·B·C)
2. 1단계 DAG: 변수 5~7개 + 화살표
3. 2단계 CLD: 강화·균형 루프 각 1개 이상
4. 3단계 베이지안: 사전·시범 결과·사후 표
5. 4단계 시나리오: 2축 매트릭스 + 4분면 권고
6. 5단계 몬테카를로: 5년 후 결과 분포 + 목표 달성 확률
7. 마지막: 5개 답을 하나의 통합 권고로 묶기

각 단계마다 **AI에게 보낼 요청 1개**를 설계하고, 결과 코드를 빈 셀에 붙여 넣어 실행한다.

**오류가 났을 때**

```text
아래 코드를 실행했더니 오류가 났어.
변수명은 my_dag, my_cld, my_bayes, my_scenarios, my_montecarlo로 통일해줘.
한글 폰트 설정도 포함해줘.

[오류 메시지]
[현재 코드]
```


---

## 10. 제출물

다음 표를 작성하여 제출한다. 결과만 내지 말고, **각 도구가 어떤 답을 줬고 왜 그렇게 통합했는지** 함께 적는다.

| 항목 | 작성 내용 |
|------|----------|
| 정책 주제 | |
| 정책 옵션 3개 (A/B/C) | |
| 1단계 DAG: 식별한 교란변수 1개 | |
| 2단계 CLD: 강화 루프 1개 + 균형 루프 1개 | |
| 3단계 베이지안: 사전 → 사후 변화 (옵션 A/B/C) | |
| 4단계 시나리오: 2축 + 4분면 권고 1줄씩 | |
| 5단계 몬테카를로: 평균·5분위·95분위·목표 달성 확률 | |
| 5개 답이 충돌한 지점 1개 | |
| 통합 권고안 (3~5줄) | |
| 가장 중요한 가정 1개 + 그 가정이 깨졌을 때 대응 | |

## 오늘의 핵심

다섯 도구는 각자 다른 진실을 본다. **DAG**는 인과 구조, **CLD**는 시간 흐름, **베이지안**은 믿음 갱신, **시나리오**는 미래 불확실성, **몬테카를로**는 결과 분포다.

하나의 도구가 답을 주는 게 아니라, **다섯 답이 충돌하는 지점이 의사결정 지점이다**. 이번 워크숍의 결과는 14주차 발표·15주차 기말고사·기말 프로젝트의 골격이 된다.
